# An in-context mixture of acquisition functions: does a budget- and context-dependent mixture beat every arm?

## Description

<!-- Written 2026-11-13 11:35 before the first run (pre-registration). With Tim, 2026-11-13: solve the convex-hull problem (an in-context mixture of the arms) directly, as the starting point for anything beyond it; sample efficiency counts beside the final regret. Follows notebooks 13 (INCONCLUSIVE) and 15 (NO-GO). Issue #13. -->

**Question.** No single arm is best on every task of the prior. **Is there a policy inside the arms' convex hull, a mixture whose weights depend on the remaining budget and on the context read from the frozen PFN, that beats every single arm on fresh tasks, in final regret and in how fast it gets there? How much of that is the budget, and how much the context?**

Why this and not another value model: notebooks 13 and 15 learned per-state advantages of options from single noisy outcomes; the value models explained 4% of the label variance and the policies collapsed to a static schedule. Here nothing is predicted. Whole policies are compared on whole runs, on the same tasks and candidate sets, and the search follows the paired difference.

**Setup** (`experimental/mixture.py`, `tasks/mixture_policy.py`). Frozen `pfn_large`, fp32, noise-free prior tasks, deterministic candidate sets (512 scrambled Sobol points + 32 perturbations of the incumbent at each of σ = 0.01, 0.05, 0.2; half the size of notebooks 12–15, for speed). Budget T = 20 steps. Two members: **d = 2** (4 initial points) and **d = 4** (8 initial points: the harder problem with a tight budget).
- *Policy.* One PFN pass scores the candidates under EI, PI(0.01), UCB(0.995), MES and the mean. Each arm's scores are normalized within the state: (score − best) / (best − median), so its best candidate is 0 and its median −1. The pick is the argmax of Σₖ wₖ · score_k. With all weight on one arm this is that arm's pick exactly (tested); mixed weights can pick a point that no arm ranks first.
- *Weights.* w = softmax(θ·φ), φ = [1, τ, τ², context features], τ = share of the budget used. 11 context features from the same pass: largest EI and PI, predictive sd near the incumbent and globally, the expected gap to the optimum's value (MES's f\* samples), the incumbent's value, the distance of EI's and UCB's picks to the incumbent, whether EI's pick is a local perturbation, steps since the last improvement, the gap between the two best values. Standardized on states of the uniform mixture. 5 × 14 parameters.
- *Objective* J per task (lower = better): ½ · log10(R_T + ε) + ½ · mean over steps of log10(R_t + ε), with **ε = 1e-3, one PFN bin**. The first half is the final regret, the second the area under the log-regret curve (efficiency). Both are reported separately, also at ε = 1e-4 and 1e-6.
- *Search.* Antithetic perturbations θ ± σe (6 pairs per generation, σ = 0.3), both sides run on the same 128 training tasks (512 in rotation; split 4, offset 80000). Runs are deterministic, so the paired difference is exact; its mean times e is the step direction (Adam). Every 5 generations the current θ is compared with the accepted one on 512 validation tasks (offset 85000) and accepted only if it is better: the accepted policy never gets worse on validation.
  - **Phase 1, budget-only:** only the constant and τ terms. Start: weight 0.83 on the arm that is best on validation.
  - **Phase 2, context:** all terms, starting from phase 1's accepted θ. 50 generations each.
- *Test.* 1536 fresh tasks (offset 90000), under candidate seed 0 (the one trained on) and seed 1 (another candidate set, a luck replicate): every arm, the uniform mixture, the budget-only policy, the context policy.

**What to look for.**
1. **Primary:** the context policy against the best single arm (chosen on validation), on J at ε = 1e-3, seed 0; paired over test tasks, bootstrap CI.
2. The context policy against the budget-only policy: the context-dependent part.
3. The budget-only policy against the best arm: what a learned schedule alone gives.
4. Final regret and area separately, at all three floors; the regret curves; a plot of final regret against area for all policies (is the learned policy on the Pareto front?).
5. Seed 1: does it hold on another candidate set?
6. What was learned: weights along the run, and θ on the context features.
7. d = 4 against d = 2: is the context worth more where the problem is harder?

**GO / NO-GO** (rule `analysis/option_policy.py::policy_verdict` on the context policy, J at ε = 1e-3, seed 0; worthwhile = 0.022 log10).
- **GO**: the context policy beats the best single arm (CI > 0), and beats the budget-only policy (CI > 0, ≥ 0.022). A context-reading policy inside the hull beats the arms and a learned schedule.
- **NO-GO**: the upper CI bound of context over budget-only is < 0.022. The context, read through these features and used linearly, adds nothing to a schedule.
- **INCONCLUSIVE**: otherwise.
- Reported beside the verdict: whether the budget-only policy beats the best arm (CI > 0). That alone is a budget-aware policy that beats every arm, but not the context-dependent one we are after.

**Limits.**
- Inside the convex hull of five arms' normalized scores, linear in 11 hand-made features; not the acquisition branch of decision 0009.
- The floor at 1e-3 makes the objective blind to polishing below one PFN bin, on purpose; the 1e-6 numbers show what that hides.
- The search is small (50 + 50 generations) and local, started at the best arm; a null result bounds this search, not the class.
- Validation tasks are used for acceptance and for choosing the best arm; the test tasks are used once.
- T = 20 at d = 4 is a tight budget chosen to make exploration matter; other budgets are not covered.
- f\* at d = 4 is coarser (notebook 12's addendum); with ε = 1e-3 this does not matter.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from anytime_acq.analysis import load_snapshot, show_header
from anytime_acq.analysis.option_policy import policy_verdict
from anytime_acq.analysis.stats import bootstrap_ci

SNAPSHOT = "results/mixture_policy/<sweep_id>"  # set after `just snapshot`
snap = load_snapshot(SNAPSHOT)
show_header(snap)  # how it was run + configuration, generated from the snapshot

In [ ]:
runs = snap.runs
print("member status:", runs.status.value_counts().to_dict())
dmap = dict(zip(runs.member, runs["task.d"])) if "task.d" in runs else {m: 2 for m in runs.member}
def tab(name):
    t = snap.table(name)
    return t.assign(d=t.member.map(dmap)) if "member" in t else t.assign(d=list(dmap.values())[0])
train, theta, per, curves, weights, summ = (tab(k) for k in ("train", "theta", "test_per_task", "test_curves", "weights", "summary"))
DS = sorted(per.d.unique()); ARMS = [p for p in per.policy.unique() if p not in ("uniform", "budget", "context")]
display(runs[["member"] + [c for c in runs.columns if c.startswith(("time/", "val_J/", "best_arm"))]].round(3))

## 1. The search

In [ ]:
fig, ax = plt.subplots(1, len(DS), figsize=(6 * len(DS), 3.2), dpi=90, squeeze=False)
for a, d in zip(ax[0], DS):
    g = train[train.d == d].reset_index(drop=True)
    a.plot(g.index, g.J_train, ".", ms=3, color="gray", label="training batch (mean of the perturbed policies)")
    a.plot(g.index, g.J_accepted, "k-", label="accepted policy on validation")
    v = g.dropna(subset=["J_val"]); a.plot(v.index, v.J_val, "o", ms=3, color="C1", label="current policy on validation")
    a.axvline(len(g[g.phase == "budget"]), color="k", ls=":"); a.set_title(f"d = {d}: objective J (lower = better); left: budget-only, right: context")
    a.set_xlabel("generation"); a.grid(alpha=.3); a.legend(fontsize=7)
plt.show()

## 2. Against the arms on fresh tasks

In [ ]:
def show(metric, seed):
    s = summ[(summ.metric == metric) & (summ.seed == seed) & (summ.contrast != "level")]
    return s.assign(ci=[f"{m:+.3f} [{lo:+.3f}, {hi:+.3f}]" for m, lo, hi in zip(s["mean"], s.lo, s.hi)], policy=s.variant.str.split(":").str[0], vs=s.contrast + " (" + s.ref + ")").pivot(index=["d", "policy"], columns="vs", values="ci")
print("objective J at eps = 1e-3 (log10, positive = better than the comparator); paired over test tasks, 95% bootstrap CI")
print("seed 0:"); display(show("J", 0)); print("seed 1 (another candidate set):"); display(show("J", 1))
print("final regret only / area under the curve only (seed 0):"); display(show("final", 0)); display(show("auc", 0))
print("levels of every policy (mean over test tasks, seed 0; lower = better):")
lv = per[per.seed == 0].groupby(["d", "policy"])[[c for c in per.columns if "@" in c]].mean().round(3)
display(lv)

In [ ]:
fig, ax = plt.subplots(2, len(DS), figsize=(6 * len(DS), 7), dpi=90, squeeze=False)
for j, d in enumerate(DS):
    c = curves[(curves.d == d) & (curves.seed == 0) & (curves.eps == 1e-3)]
    for p, g in c.groupby("policy"):
        ax[0, j].plot(g.step, g.log_regret, lw=2.2 if p in ("budget", "context") else 1, ls="--" if p == "uniform" else "-", label=p)
    ax[0, j].set_title(f"d = {d}: mean log10(regret + 1e-3) along the run"); ax[0, j].set_xlabel("step"); ax[0, j].legend(fontsize=7); ax[0, j].grid(alpha=.3)
    m = per[(per.d == d) & (per.seed == 0)].groupby("policy")[["final@0.001", "auc@0.001"]].mean()
    for p, r in m.iterrows():
        ax[1, j].plot(r["auc@0.001"], r["final@0.001"], "o" if p not in ("budget", "context") else "*", ms=8 if p not in ("budget", "context") else 14); ax[1, j].annotate(p, (r["auc@0.001"], r["final@0.001"]), fontsize=8, xytext=(4, 4), textcoords="offset points")
    ax[1, j].set_xlabel("area: mean over steps of log10 regret (lower = sooner)"); ax[1, j].set_ylabel("final log10 regret"); ax[1, j].set_title(f"d = {d}: final regret against area (lower left = better)"); ax[1, j].grid(alpha=.3)
plt.tight_layout(); plt.show()

## 3. What was learned: weights along the run, and the context terms

In [ ]:
fig, ax = plt.subplots(len(DS), 2, figsize=(11, 3 * len(DS)), dpi=90, squeeze=False)
for i, d in enumerate(DS):
    for a, p in zip(ax[i], ("budget", "context")):
        weights[(weights.d == d) & (weights.policy == p)].pivot(index="step", columns="arm", values="weight").plot.area(ax=a, lw=0)
        a.set_title(f"d = {d}, {p} policy: mean weight per arm along the run (test tasks)"); a.set_ylim(0, 1); a.legend(fontsize=7, ncol=5, loc="lower center")
plt.tight_layout(); plt.show()
for d in DS:
    print(f"d = {d}: theta of the context policy (rows: phi terms; a positive entry moves weight to that arm when the term is high)")
    display(theta[(theta.d == d) & (theta.policy == "context")].pivot(index="term", columns="arm", values="theta").round(2))

## Verdict

The GO / NO-GO criterion from the description, evaluated, in one line. Record it with `just decide`.

In [ ]:
for d in DS:
    s = summ[(summ.d == d) & (summ.metric == "J")]
    b = s[(s.variant == "budget") & (s.seed == 0) & (s.contrast == "vs_base")].iloc[0]
    print(f"d = {d}: pre-registered rule on the context policy: {policy_verdict(s, 'context', 0.022)};"
          f" budget-only over the best arm ({b.ref}): {b['mean']:+.3f} [{b.lo:+.3f}, {b.hi:+.3f}]")